In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: HS, HRPT

This example demonstrates a Rietveld refinement of HS crystal
structure using constant wavelength neutron powder diffraction data
from HRPT at PSI.

## 🛠️ Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='hs')

### Set Space Group

In [4]:
structure.space_group.name_h_m = 'R -3 m'
structure.space_group.coord_system_code = 'h'

### Set Unit Cell

In [5]:
structure.cell.length_a = 6.9
structure.cell.length_c = 14.1

### Set Atom Sites

In [6]:
structure.atom_sites.create(
    id='Zn',
    type_symbol='Zn',
    fract_x=0,
    fract_y=0,
    fract_z=0.5,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='Cu',
    type_symbol='Cu',
    fract_x=0.5,
    fract_y=0,
    fract_z=0,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='O',
    type_symbol='O',
    fract_x=0.21,
    fract_y=-0.21,
    fract_z=0.06,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='Cl',
    type_symbol='Cl',
    fract_x=0,
    fract_y=0,
    fract_z=0.197,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='H',
    type_symbol='2H',
    fract_x=0.13,
    fract_y=-0.13,
    fract_z=0.08,
    adp_iso=0.5,
)

## 🔬 Define Experiment

This section shows how to add experiments, configure their parameters,
and link the structures defined in the previous step.

### Download Data

In [7]:
data_path = download_data('meas-hs-hrpt', destination='data')

Getting data...


Data 'meas-hs-hrpt': HS, HRPT (PSI)


✅ Data 'meas-hs-hrpt' downloaded to '../../../data/meas-hs-hrpt.xye'


### Create Experiment

In [8]:
expt = ExperimentFactory.from_data_path(name='hrpt', data_path=data_path)

### Set Instrument

In [9]:
expt.instrument.setup_wavelength = 1.89
expt.instrument.calib_twotheta_offset = 0.0

### Set Peak Profile

In [10]:
expt.peak.show_supported()
expt.peak.type = 'pseudo-voigt + berar-baldinozzi asymmetry'
expt.peak.broad_gauss_u = 0.1
expt.peak.broad_gauss_v = -0.2
expt.peak.broad_gauss_w = 0.2
expt.peak.broad_lorentz_x = 0.0
expt.peak.broad_lorentz_y = 0

Peak types


,,Type,Description
1,*,pseudo-voigt,CWL pseudo-Voigt profile
2,,pseudo-voigt + berar-baldinozzi asymmetry,CWL pseudo-Voigt profile with Berar-Baldinozzi asymmetry correction.


⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • asym_beba_a0=0.0                                                                                                             
   • asym_beba_a1=0.0                                                                                                             
   • asym_beba_b0=0.0                                                                                                             
   • asym_beba_b1=0.0                                                                                                             


Peak profile type for experiment 'hrpt' changed to


pseudo-voigt + berar-baldinozzi asymmetry


### Set Background

In [11]:
expt.background.auto_estimate()

### Set Linked Structures

In [12]:
expt.linked_structures.create(structure_id='hs', scale=0.5)

## 📦 Define Project

The project object is used to manage the structure, experiment, and
analysis.

### Create Project

In [13]:
project = Project(name='hs_hrpt')

### Add Structure

In [14]:
project.structures.add(structure)

### Add Experiment

In [15]:
project.experiments.add(expt)

## 🚀 Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.


### Display Structure

In [16]:
project.display.structure(struct_name='hs')

Structure 🧩 'hs' (Atom view type: 'covalent')


### Display Pattern

In [17]:
project.display.pattern(expt_name='hrpt')

In [18]:
project.display.pattern(expt_name='hrpt', x_min=48, x_max=51)

### Perform Fit 1/4

Set parameters to be refined.

In [19]:
structure.cell.length_a.free = True
structure.cell.length_c.free = True

expt.linked_structures['hs'].scale.free = True
expt.instrument.calib_twotheta_offset.free = True

Show free parameters after selection.

In [20]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,hs,cell,,length_a,6.90000,,-inf,inf,Å
2,hs,cell,,length_c,14.10000,,-inf,inf,Å
3,hrpt,linked_structure,hs,scale,0.50000,,-inf,inf,
4,hrpt,instrument,,twotheta_offset,0.00000,,-inf,inf,deg


#### Run Fitting

In [21]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.07,488.37,
2,8,0.53,119.76,75.5% ↓
3,13,1.03,112.72,5.9% ↓
4,18,1.33,107.23,4.9% ↓
5,23,1.64,104.60,2.4% ↓
6,28,1.95,103.23,1.3% ↓
7,33,2.26,102.15,1.0% ↓
8,38,2.57,101.08,1.0% ↓
9,43,2.88,99.91,1.2% ↓
10,48,3.41,98.56,1.4% ↓


🏆 Best goodness-of-fit (reduced χ²) is 62.91 at iteration 313


✅ Fitting complete.


In [22]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),21.45
4,🔁 Iterations,311
5,📏 Goodness-of-fit (reduced χ²),62.91
6,"📏 R-factor (Rf, %)",20.29
7,"📏 R-factor squared (Rf², %)",33.17
8,"📏 Weighted R-factor (wR, %)",32.69


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,hs,cell,,length_a,Å,6.9000,6.8624,0.0004,0.54 % ↓
2,hs,cell,,length_c,Å,14.1000,14.1403,0.0010,0.29 % ↑
3,hrpt,linked_structure,hs,scale,,0.5000,0.2541,0.0038,49.17 % ↓
4,hrpt,instrument,,twotheta_offset,deg,0.0000,0.1344,0.0061,N/A


#### Display Pattern

In [23]:
project.display.pattern(expt_name='hrpt')

In [24]:
project.display.pattern(expt_name='hrpt', x_min=48, x_max=51)

### Perform Fit 2/4

Set more parameters to be refined.

In [25]:
expt.peak.broad_gauss_u.free = True
expt.peak.broad_gauss_v.free = True
expt.peak.broad_gauss_w.free = True
expt.peak.broad_lorentz_y.free = True

for point in expt.background:
    point.intensity.free = True

Show free parameters after selection.

In [26]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,hs,cell,,length_a,6.86240,0.00035,-inf,inf,Å
2,hs,cell,,length_c,14.14033,0.00097,-inf,inf,Å
3,hrpt,linked_structure,hs,scale,0.25414,0.00381,-inf,inf,
4,hrpt,peak,,broad_gauss_u,0.10000,,-inf,inf,deg²
5,hrpt,peak,,broad_gauss_v,-0.20000,,-inf,inf,deg²
6,hrpt,peak,,broad_gauss_w,0.20000,,-inf,inf,deg²
7,hrpt,peak,,broad_lorentz_y,0.00000,,-inf,inf,deg
8,hrpt,instrument,,twotheta_offset,0.13441,0.00606,-inf,inf,deg
9,hrpt,background,1,intensity,645.00000,,-inf,inf,
10,hrpt,background,2,intensity,460.00000,,-inf,inf,


#### Run Fitting

In [27]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.08,63.22,
2,24,1.84,31.68,49.9% ↓
3,45,3.66,30.40,4.0% ↓
4,66,5.52,28.72,5.5% ↓
5,87,7.28,28.31,1.4% ↓
6,145,12.28,28.30,
7,209,17.32,28.30,
8,235,19.63,28.30,


🏆 Best goodness-of-fit (reduced χ²) is 28.30 at iteration 234


✅ Fitting complete.


In [28]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),19.63
4,🔁 Iterations,232
5,📏 Goodness-of-fit (reduced χ²),28.30
6,"📏 R-factor (Rf, %)",12.98
7,"📏 R-factor squared (Rf², %)",19.12
8,"📏 Weighted R-factor (wR, %)",16.94


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,hs,cell,,length_a,Å,6.8624,6.8616,0.0006,0.01 % ↓
2,hs,cell,,length_c,Å,14.1403,14.1383,0.0016,0.01 % ↓
3,hrpt,linked_structure,hs,scale,,0.2541,0.4474,0.0065,76.04 % ↑
4,hrpt,peak,,broad_gauss_u,deg²,0.1000,0.4739,0.0468,373.88 % ↑
5,hrpt,peak,,broad_gauss_v,deg²,-0.2000,-0.4252,0.0824,112.60 % ↑
6,hrpt,peak,,broad_gauss_w,deg²,0.2000,0.2541,0.0335,27.05 % ↑
7,hrpt,peak,,broad_lorentz_y,deg,0.0000,0.1782,0.0155,N/A
8,hrpt,instrument,,twotheta_offset,deg,0.1344,0.1215,0.0063,9.62 % ↓
9,hrpt,background,1,intensity,,645.0000,622.6100,29.9847,3.47 % ↓
10,hrpt,background,2,intensity,,460.0000,444.2618,9.3318,3.42 % ↓


#### Display Pattern

In [29]:
project.display.pattern(expt_name='hrpt')

In [30]:
project.display.pattern(expt_name='hrpt', x_min=48, x_max=51)

### Perform Fit 3/4

Set more parameters to be refined.

In [31]:
structure.atom_sites['O'].fract_x.free = True
structure.atom_sites['O'].fract_z.free = True
structure.atom_sites['Cl'].fract_z.free = True
structure.atom_sites['H'].fract_x.free = True
structure.atom_sites['H'].fract_z.free = True

Show free parameters after selection.

In [32]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,hs,cell,,length_a,6.86156,0.00060,-inf,inf,Å
2,hs,cell,,length_c,14.13831,0.00161,-inf,inf,Å
3,hs,atom_site,O,fract_x,0.21000,,-inf,inf,
4,hs,atom_site,O,fract_z,0.06000,,-inf,inf,
5,hs,atom_site,Cl,fract_z,0.19700,,-inf,inf,
6,hs,atom_site,H,fract_x,0.13000,,-inf,inf,
7,hs,atom_site,H,fract_z,0.08000,,-inf,inf,
8,hrpt,linked_structure,hs,scale,0.44738,0.00653,-inf,inf,
9,hrpt,peak,,broad_gauss_u,0.47388,0.04678,-inf,inf,deg²
10,hrpt,peak,,broad_gauss_v,-0.42520,0.08238,-inf,inf,deg²


#### Run Fitting

In [33]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.08,28.34,
2,29,2.43,20.77,26.7% ↓
3,55,4.63,20.22,2.6% ↓
4,113,9.64,20.20,
5,172,14.64,20.20,
6,235,19.67,20.20,
7,238,20.12,20.20,


🏆 Best goodness-of-fit (reduced χ²) is 20.20 at iteration 237


✅ Fitting complete.


In [34]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),20.12
4,🔁 Iterations,235
5,📏 Goodness-of-fit (reduced χ²),20.20
6,"📏 R-factor (Rf, %)",10.29
7,"📏 R-factor squared (Rf², %)",16.48
8,"📏 Weighted R-factor (wR, %)",14.68


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,hs,cell,,length_a,Å,6.8616,6.8611,0.0005,0.01 % ↓
2,hs,cell,,length_c,Å,14.1383,14.1402,0.0013,0.01 % ↑
3,hs,atom_site,O,fract_x,,0.2100,0.2064,0.0005,1.69 % ↓
4,hs,atom_site,O,fract_z,,0.0600,0.0579,0.0003,3.57 % ↓
5,hs,atom_site,Cl,fract_z,,0.1970,0.1963,0.0004,0.34 % ↓
6,hs,atom_site,H,fract_x,,0.1300,0.1342,0.0004,3.20 % ↑
7,hs,atom_site,H,fract_z,,0.0800,0.0896,0.0003,12.05 % ↑
8,hrpt,linked_structure,hs,scale,,0.4474,0.4409,0.0054,1.44 % ↓
9,hrpt,peak,,broad_gauss_u,deg²,0.4739,0.3764,0.0310,20.57 % ↓
10,hrpt,peak,,broad_gauss_v,deg²,-0.4252,-0.3756,0.0567,11.66 % ↓


#### Display Pattern

In [35]:
project.display.pattern(expt_name='hrpt')

In [36]:
project.display.pattern(expt_name='hrpt', x_min=48, x_max=51)

### Perform Fit 4/4

Set more parameters to be refined.

In [37]:
structure.atom_sites['Zn'].adp_iso.free = True
structure.atom_sites['Cu'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True
structure.atom_sites['Cl'].adp_iso.free = True
structure.atom_sites['H'].adp_iso.free = True

expt.peak.asym_beba_a0.free = True
expt.peak.asym_beba_b0.free = True
expt.peak.asym_beba_a1.free = True
expt.peak.asym_beba_b1.free = True

Show free parameters after selection.

In [38]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,hs,cell,,length_a,6.86113,0.00049,-inf,inf,Å
2,hs,cell,,length_c,14.14024,0.00125,-inf,inf,Å
3,hs,atom_site,Zn,adp_iso,0.50000,,-inf,inf,Å²
4,hs,atom_site,Cu,adp_iso,0.50000,,-inf,inf,Å²
5,hs,atom_site,O,fract_x,0.20644,0.00051,-inf,inf,
6,hs,atom_site,O,fract_z,0.05786,0.00035,-inf,inf,
7,hs,atom_site,O,adp_iso,0.50000,,-inf,inf,Å²
8,hs,atom_site,Cl,fract_z,0.19634,0.00039,-inf,inf,
9,hs,atom_site,Cl,adp_iso,0.50000,,-inf,inf,Å²
10,hs,atom_site,H,fract_x,0.13416,0.00038,-inf,inf,


#### Run Fitting

In [39]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.08,20.26,
2,38,3.17,19.49,3.8% ↓
3,73,5.89,19.05,2.3% ↓
4,128,10.94,18.99,
5,187,15.98,18.98,
6,249,21.03,18.98,
7,308,26.05,18.98,
8,366,31.09,18.98,
9,425,36.15,18.98,
10,487,41.16,18.98,


⚠️ Parameter 'hs.atom_site.Zn.adp_iso' (-1.43049295) is below its physical lower limit (0.0).                                     


🏆 Best goodness-of-fit (reduced χ²) is 18.98 at iteration 493


✅ Fitting complete.


In [40]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),42.55
4,🔁 Iterations,491
5,📏 Goodness-of-fit (reduced χ²),18.98
6,"📏 R-factor (Rf, %)",9.82
7,"📏 R-factor squared (Rf², %)",15.63
8,"📏 Weighted R-factor (wR, %)",13.68


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,hs,cell,,length_a,Å,6.8611,6.8580,0.0013,0.05 % ↓
2,hs,cell,,length_c,Å,14.1402,14.1351,0.0028,0.04 % ↓
3,hs,atom_site,Zn,adp_iso,Å²,0.5000,-1.4305,0.1617,386.10 % ↓
4,hs,atom_site,Cu,adp_iso,Å²,0.5000,1.6785,0.1484,235.69 % ↑
5,hs,atom_site,O,fract_x,,0.2064,0.2082,0.0005,0.85 % ↑
6,hs,atom_site,O,fract_z,,0.0579,0.0584,0.0004,0.93 % ↑
7,hs,atom_site,O,adp_iso,Å²,0.5000,0.5582,0.1546,11.64 % ↑
8,hs,atom_site,Cl,fract_z,,0.1963,0.1969,0.0004,0.27 % ↑
9,hs,atom_site,Cl,adp_iso,Å²,0.5000,0.1683,0.1236,66.34 % ↓
10,hs,atom_site,H,fract_x,,0.1342,0.1339,0.0005,0.17 % ↓


In [41]:
project.display.fit.correlations()

#### Display Pattern

In [42]:
project.display.pattern(expt_name='hrpt')

In [43]:
project.display.pattern(expt_name='hrpt', x_min=48, x_max=51)

## 📊 Report

The HTML report is written automatically when the project is saved;
enable `project.report.pdf` as well for a PDF version.

## 💾 Save Project

In [44]:
project.save_as(dir_path='projects/refine-hs-hrpt')

Saving project 📦 'hs_hrpt' to '../../../projects/refine-hs-hrpt'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 hs.edi


├── 📁 experiments/


│   └── 📄 hrpt.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 hs_hrpt.html
